# Movie Sentiments Dashboard

In [487]:
import subprocess, sys
for pkg in ['dash', 'dash-bootstrap-components', 'plotly', 'pandas', 'nbformat']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    
import pandas as pd
import plotly.graph_objects as go

print("Packages installed")

Packages installed


In [488]:
# ── Data ──────────────────────────────────────────────────────

movies_df = pd.read_csv('tmdb_movie_enriched_scl_aggregate.csv')

movies_df['release_date'] = pd.to_datetime(movies_df['release_date'], errors='coerce')
movies_df['year'] = movies_df['release_date'].dt.year
movies_df['revenue'] = pd.to_numeric(movies_df['revenue'], errors='coerce')
movies_df['sentiment_mean'] = pd.to_numeric(movies_df['sentiment_mean'], errors='coerce')

# Movie Data Filter --------------------------------------------------------------------
movies_df = movies_df[
    movies_df['year'].between(1910, 2025) &
    (movies_df['revenue'] > 0) &
    (movies_df['vote_count'] >= 10) &
    movies_df['dominant_polarity'].notna()
].copy()

movies_df['sentiment_category'] = movies_df['dominant_polarity']

print(f'{len(movies_df):,} movies')
movies_df['sentiment_category'].value_counts()

11,541 movies


sentiment_category
positive    6518
negative    4966
neutral       57
Name: count, dtype: int64

In [489]:
# Theme & Colors --------------------------------------------------------------------
COLORS = {
    'positive': '#FF7A30',
    'negative': '#3D74B6',
    'neutral': '#D6D4CC',
    'unknown': '#A0A0A0',
    'bg': '#DDDDDD',
    'bg-light': '#f3f6f4',
    'text': '#36454F',
    'grid': 'rgba(0,0,0,0.08)',
}

layout_base = dict(
    plot_bgcolor=COLORS['bg-light'],
    paper_bgcolor=COLORS['bg'],
    font=dict(family='Inter, sans-serif', size=13, color=COLORS['text']),
    hovermode='x unified',
    showlegend=False,
    margin=dict(l=80, r=40, t=100, b=60),
)

In [490]:
def make_area_chart(df, sentiments=['positive', 'negative'], title='Positive Films Earn More at the Box Office'):
    data = (df.groupby(['year', 'sentiment_category'])['revenue']
              .sum()
              .unstack(fill_value=0)
              .reindex(columns=sentiments, fill_value=0))
    smoothed = data.rolling(5, min_periods=1, center=True).mean()
    stack_order = ['negative', 'positive']
    fig = go.Figure()
    for s in stack_order:
        if s not in sentiments:
            continue
        fig.add_trace(go.Scatter(
            x=smoothed.index, y=smoothed[s],
            name=s.capitalize(),
            mode='lines',
            line=dict(width=0.5, color=COLORS[s]),
            stackgroup='one',
            groupnorm='percent', #https://plotly.com/python/filled-area-plots/
        ))

    fig.update_layout(title=dict(text=title, x=0.5, font=dict(size=20, color=COLORS['text'])),
                      **layout_base, height=600)
    fig.add_annotation(text='Positive', xref='paper', yref='paper',
                        x=0.95, y=0.85, showarrow=False,
                        font=dict(size=14, color=COLORS['text']))
    fig.add_annotation(text='Negative', xref='paper', yref='paper',
                        x=0.95, y=0.15, showarrow=False,
                        font=dict(size=14, color=COLORS['text']))

    fig.update_xaxes(title='Year', showgrid=True, gridcolor=COLORS['grid'], zeroline=False)
    fig.update_yaxes(title='% of Revenue', showgrid=True, gridcolor=COLORS['grid'],
                     zeroline=False, range=[0, 100], ticksuffix='%')
    return fig
fig = make_area_chart(movies_df)
fig.show()

In [491]:
NUM_GENRES = 6

def make_genre_chart(df, year_range):
    exploded = df.assign(genre=df['genres'].str.split(', ')).explode('genre')
    top_genres = exploded['genre'].value_counts().head(NUM_GENRES).index.tolist()
    exploded = exploded[exploded['genre'].isin(top_genres)]

    counts = exploded.groupby(['genre', 'sentiment_category']).size().unstack(fill_value=0)
    counts = counts.reindex(columns=['positive', 'negative'], fill_value=0)
    totals = counts.sum(axis=1)
    pcts = counts.div(totals, axis=0) * 100

    pcts = pcts.loc[top_genres]
    counts = counts.loc[top_genres]

    pcts['negative'] = pcts['negative'] * -1

    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=pcts.index, x=pcts['negative'],
        name='Negative', orientation='h',
        marker=dict(color=COLORS['negative'], opacity=0.9),
        text=[f"{c} ({p:.0f}%)" for c, p in zip(counts['negative'], counts['negative'] / totals.loc[top_genres] * 100)],
        textposition='inside',
        hoverinfo='skip',
        legendrank=1,
    ))
    fig.add_trace(go.Bar(
        y=pcts.index, x=pcts['positive'],
        name='Positive', orientation='h',
        marker=dict(color=COLORS['positive'], opacity=0.9),
        text=[f"{c} ({p:.0f}%)" for c, p in zip(counts['positive'], counts['positive'] / totals.loc[top_genres] * 100)],
        textposition='inside',
        hoverinfo='skip',
        legendrank=2,
    ))

    title = f"Genre Tug-of-War ({year_range[0]}–{year_range[1]})"
    fig.update_layout(**layout_base)
    fig.update_layout(
        title=dict(text=title, x=0.5, font=dict(size=18, color=COLORS['text'])),
        barmode='relative',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
        yaxis=dict(autorange='reversed'),
        height=450,
    )
    fig.update_xaxes(
        zeroline=True,
        zerolinecolor='black',
        zerolinewidth=1,
        tickmode='array',
        tickvals=[-100, -50, -25, 0, 25,50, 100],
        ticktext=['100%', '25%', '50%', '0', '25%', '50%', '100%'],
        showgrid=False,
    )
    return fig

fig = make_genre_chart(movies_df, [1936, 2023])
fig.show()
# diverging bar chart https://plotly.com/python/horizontal-bar-charts/

In [ ]:
# Layout --------------------------------------------------------------------
from dash import Dash, dcc, html, Input, Output
import dash_bootstrap_components as dbc

MIN_YEAR, MAX_YEAR = int(movies_df['year'].min()), int(movies_df['year'].max())

def create_layout():
    return dbc.Container([

        html.H2('Feel Good Films: Movie Sentiment & Box Office Revenue', style={'textAlign': 'center', 'padding': '20px 0'}),

        # Filter Card
        dbc.Card([
            dbc.CardBody([
                html.Label('Year Range'),
                dcc.RangeSlider(
                    id='year-slider',
                    min=MIN_YEAR, max=MAX_YEAR, step=1,
                    value=[1960, MAX_YEAR],
                    marks={y: str(y) for y in range(MIN_YEAR, MAX_YEAR + 1, 20)},
                ),
                html.Hr(),
                html.Label('Sentiment Filter'),
                dcc.Checklist(
                    id='sentiment-filter',
                    options=[
                        {'label': ' Positive', 'value': 'positive'},
                        {'label': ' Negative', 'value': 'negative'},
                    ],
                    value=['positive', 'negative'],
                    inline=True,
                ),
            ]),
        ], style={'marginBottom': '20px'}),

        # Revenue / Sentiment Stacked Area Chart
        dcc.Graph(id='area-chart'),

        # Genre Tug of War Chart
        dbc.Row([
            dbc.Col(dcc.Graph(id='genre-chart'), width=6),
        ])

    ], fluid=True, style={'backgroundColor': COLORS['bg'], 'padding': '20px', '--rc-slider-color': '#333'})

In [493]:
# Register Callbacks --------------------------------------------------------------------─
def register_callbacks(app):
    @app.callback(
        Output('area-chart', 'figure'),
        Output('genre-chart', 'figure'),
        Input('year-slider', 'value'),
        Input('sentiment-filter', 'value'),
    )
    def update_charts(year_range, sentiments):
        filtered = movies_df[movies_df['year'].between(year_range[0], year_range[1])]
        return (
            make_area_chart(filtered, sentiments=sentiments),
            make_genre_chart(filtered, year_range),
        )

In [494]:
# Create app & run --------------------------------------------------------------------
EXT = [
    dbc.themes.LUX,
    "https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0/css/all.min.css",
]

app = Dash(__name__, external_stylesheets=EXT, suppress_callback_exceptions=True)
app.title = "Movie Keyword Sentiment and Box Office Revenue"
app.layout = create_layout()
register_callbacks(app)

app.run(jupyter_mode='inline', debug=False)